# Analysing Pharmaceutical Sales Data

This notebook uses Pandas and Matplotlib to explore daily sales from 2014–2019. The dataset contains totals for eight ATC categories, not individual medicine or brand names, so the rankings below are at category level.

## Load and clean the daily data

Parse the date, coerce category totals to numeric values, remove incomplete and duplicate dates, and derive year/month from the parsed date.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from analyze_sales import (
    DRUG_CATEGORIES, PERIODS, average_daily_sales_by_category,
    load_sales_data, respiratory_sales_by_month, top_categories_for_period,
    total_sales_by_category, save_outputs,
)

sales = load_sales_data()
print(f'{len(sales):,} daily records: {sales.Date.min():%Y-%m-%d} to {sales.Date.max():%Y-%m-%d}')
display(sales.head())

## Total sales by ATC category

The columns identify therapeutic categories. The source does not include product-level or brand-level data, so these totals cannot be split into individual brands.

In [ ]:
category_totals = total_sales_by_category(sales)
display(category_totals.rename('Total Sales').to_frame())

ax = category_totals.sort_values().plot.barh(figsize=(9, 5), color='#3867d6')
ax.set(title='Total Sales by ATC Drug Category', xlabel='Sales quantity', ylabel='ATC category')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

## Highest categories in selected periods

The task asks for individual drug brands in these date ranges, but the available CSV has only category-level aggregates. The table below reports the top three ATC categories as the closest answer supported by the data.

In [ ]:
period_rankings = []
for period_name, (start, end) in PERIODS.items():
    ranked = top_categories_for_period(sales, start, end)
    for rank, (category, quantity) in enumerate(ranked.items(), start=1):
        period_rankings.append({'Period': period_name, 'Rank': rank, 'ATC Category': category, 'Sales': quantity})
period_table = pd.DataFrame(period_rankings)
display(period_table)

## Category with the highest sales in 2017

In [ ]:
sales_2017 = sales.loc[sales['Year'].eq(2017), DRUG_CATEGORIES].sum().sort_values(ascending=False)
display(sales_2017.head(1).rename('2017 Sales').to_frame())

## Average daily sales by category

The average includes all days in the dataset, including days with zero sales for a category.

In [ ]:
daily_averages = average_daily_sales_by_category(sales)
display(daily_averages.rename('Average Daily Sales').to_frame())

## Are respiratory medicines (R03) seasonal?

Compare the average daily R03 sales for each calendar month across the observed years, which avoids month-length differences dominating a total-sales comparison.

In [ ]:
r03_monthly_average = respiratory_sales_by_month(sales)
display(r03_monthly_average.rename('Average Daily R03 Sales').to_frame())

ax = r03_monthly_average.plot(marker='o', figsize=(9, 5), color='#20a779', linewidth=2)
ax.set_xticks(range(1, 13))
ax.set(title='Average Daily R03 Sales by Month', xlabel='Month', ylabel='Average daily sales quantity')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Save tables and charts

The script and this notebook can save the summary tables and figures under `outputs/`.

In [ ]:
save_outputs(sales)
print('Saved category totals, selected-period rankings, and charts under outputs/.')